# Track 04 — Generation, Synthesis, and Automated RAG Evaluation (RAGAS / TruLens)

> **RAG Engineering Academy — Track 04: Generation & Automated Evaluation**  
> *Core Focus: Grounded Synthesis, Citation Verification, Epistemic Refusals, The RAG Triad, RAGAS Metrics (Context Precision, Context Recall, Faithfulness, Answer Relevance), and 100% Offline Evaluation Harnesses.*

---

## 🎯 Executive Overview & Pedagogical Objectives

In earlier tracks, we built the retrieval pipeline: document ingestion, PDF parsing, semantic chunking, dense vector indexing, and precision cross-encoder reranking. However, in production systems, **retrieval is only half the battle**:

1. **The Generation Vulnerability:** An LLM presented with retrieved context can still hallucinate outside facts, extrapolate beyond provided evidence, or misattribute statements to cited sources.
2. **Context Window Distortion:** Retrieved chunks are often noisy. If irrelevant distractors are placed above relevant chunks, or if the context window is stuffed indiscriminately, model reasoning degrades significantly (*the "Lost in the Middle" effect*).
3. **The Evaluation Bottleneck:** Human evaluation does not scale to thousands of production queries. Naive text similarity metrics like BLEU or ROUGE fail because they compare n-gram surface forms rather than semantic correctness and factual grounding.

To establish production reliability, this track implements **Automated RAG Evaluation** rooted in the **RAG Triad (TruLens)** and **RAGAS (Retrieval Augmented Generation Assessment)** frameworks.

```mermaid
flowchart TD
    UserQuery["User Query"] --> ContextRetrieved["Retrieved Context Chunks"]
    ContextRetrieved --> Generator["Grounded Generator / LLM"]
    Generator --> AnswerGenerated["Generated Answer + Inline Citations"]
    
    subgraph RAG_Triad ["The RAG Triad Evaluation Pillars"]
        direction TB
        Pillar1["1. Context Precision / Relevance
(Query <-> Retrieved Context)
Is the context noise-free and relevant?"]
        Pillar2["2. Faithfulness / Groundedness
(Context <-> Generated Answer)
Are all claims supported by the context?"]
        Pillar3["3. Answer Relevance
(Query <-> Generated Answer)
Does the answer address the question directly?"]
    end
    
    UserQuery -.-> Pillar1
    ContextRetrieved -.-> Pillar1
    
    ContextRetrieved -.-> Pillar2
    AnswerGenerated -.-> Pillar2
    
    UserQuery -.-> Pillar3
    AnswerGenerated -.-> Pillar3
    
    Pillar1 --> TriadScore["Composite Harmonic Triad Score
Diagnostic Root-Cause Analysis"]
    Pillar2 --> TriadScore
    Pillar3 --> TriadScore

    classDef stage fill:#f8fafc,stroke:#3b82f6,stroke-width:1.5px,color:#1e293b;
    class UserQuery,ContextRetrieved,Generator,AnswerGenerated,Pillar1,Pillar2,Pillar3,TriadScore stage;
```

### Key Pedagogical Objectives
1. **Master Grounded Prompt Engineering:** Construct prompts with context delimiters, strict epistemic refusal instructions, and inline citation attribution.
2. **Analyze the RAG Triad Architecture:** Understand the distinct responsibilities of **Context Precision**, **Context Recall**, **Faithfulness**, and **Answer Relevance**.
3. **Decompose Text into Atomic Propositions:** Implement claim extraction to verify factual truth claim-by-claim.
4. **Implement Mathematical Metrics Offline:** Build 100% offline CPU metric calculators that penalize irrelevant top-ranked chunks and detect hallucinations without requiring external paid APIs.
5. **Pinpoint Root-Cause Failure Modes:** Distinguish between retrieval failures (low recall), context pollution (low precision), generator hallucinations (low faithfulness), and evasive drift (low relevance).
6. **Dual-Mode Execution:** Seamlessly switch between the deterministic local CPU evaluation harness and live Groq LLM-as-a-Judge evaluations.


## 1. Environment Verification and Module Imports

We initialize our local Python environment, importing common evaluation and synthesis utilities from `common.evaluation`. All evaluation metrics run 100% locally on CPU by default with zero external network requirements.


In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Any

from common.evaluation import (
    EvaluationSample,
    EvaluationResult,
    ClaimVerification,
    DeterministicSemanticEmbedder,
    PropositionExtractor,
    OfflineClaimVerifier,
    RAGSynthesizer,
    RAGEvaluationHarness,
    compute_context_precision,
    compute_context_recall,
    compute_faithfulness,
    compute_answer_relevance,
    compute_rag_triad_score,
)
from common.config import settings

print(f"System Initialized. Active LLM Provider: {settings.llm_provider}")
print(f"Groq API Key Configured: {bool(settings.groq_api_key)}")


## 2. Grounded Synthesis & Citation Attribution

A production RAG generator must adhere to three foundational engineering constraints:
1. **Context Encapsulation:** Retrieved context chunks must be demarcated with explicit boundaries and document identifiers (e.g. `[Doc 1]`, `[Doc 2]`) to prevent prompt injection and cross-document confusion.
2. **Inline Citation Attribution:** Every factual assertion in the generated answer must cite its backing chunk (`[Doc 1]`), establishing provenance and auditability.
3. **Epistemic Refusal:** If retrieved passages do not contain sufficient evidence to answer the query, the generator must explicitly refuse (`"Based on the provided context, I cannot answer this question."`) rather than speculate or hallucinate outside knowledge.

Below, we inspect how `RAGSynthesizer` formats prompt assemblies with strict epistemic boundaries.


In [ ]:
synth = RAGSynthesizer()

sample_contexts = [
    "ChromaDB stores embeddings locally using SQLite for metadata and duckdb/parquet for vector indices.",
    "HNSW builds a hierarchical graph with geometric decay factor mL = 1 / ln(M) for fast logarithmic search.",
    "PyMuPDF parses PDF documents, rendering pages and extracting text blocks with high layout fidelity.",
]

sample_query = "How does ChromaDB store its metadata and vector indices locally?"
prompt = synth.build_prompt(sample_query, sample_contexts)
print(prompt)


### Citation Parsing and Validation

Next, we demonstrate automated citation validation. If a generated answer references `[Doc 1]` and `[Doc 2]`, our validation engine checks whether those documents exist in the retrieved context pool, flagging hallucinated citations (e.g. `[Doc 99]`).


In [ ]:
valid_answer = (
    "ChromaDB persists metadata in local SQLite tables [Doc 1], while "
    "HNSW organizes vectors in hierarchical skip-list graphs [Doc 2]."
)
invalid_answer = (
    "ChromaDB uses distributed consensus for indexing [Doc 99]."
)

audit_valid = synth.validate_citations(valid_answer, num_contexts=len(sample_contexts))
audit_invalid = synth.validate_citations(invalid_answer, num_contexts=len(sample_contexts))

print("Valid Answer Audit:", audit_valid)
print("Invalid Answer Audit:", audit_invalid)


## 3. Atomic Proposition & Claim Decomposition

Evaluating factual correctness at the paragraph level is deeply flawed. Paragraphs mix true statements with subtle hallucinations. Consider:

> *"ChromaDB provides local vector storage with SQLite metadata persistence, and it automatically deploys a Kubernetes cluster on AWS."*

- First clause: **TRUE** (grounded in context).
- Second clause: **FALSE** (complete hallucination).

In modern evaluation frameworks (**FactScore, RAGAS, TruLens**), the answer is first parsed into **atomic propositions** (self-contained factual statements). Each proposition is then independently evaluated for entailment against retrieved context.


In [ ]:
extractor = PropositionExtractor()

compound_response = (
    "PyMuPDF extracts text from PDF documents with AGPL license compliance, "
    "and ChromaDB indexes the extracted vectors for fast similarity search. "
    "Furthermore, ChromaDB automatically fine-tunes BERT models on GPU clusters."
)

atomic_claims = extractor.extract_claims(compound_response)
print(f"Extracted {len(atomic_claims)} Atomic Propositions:")
for i, claim in enumerate(atomic_claims, start=1):
    print(f"  [{i}] {claim}")


## 4. The 4 Core Evaluation Metrics: Formulations and Implementation

We now implement and examine the four canonical RAG evaluation dimensions:

---

### 4.1 Context Precision (Signal-to-Noise Ratio & Rank Sensitivity)

**Definition:** Evaluates whether retrieved chunks are relevant to answering the query, and heavily penalizes irrelevant chunks ranked above relevant ones.

$$\text{Context Precision@K} = \frac{1}{\sum_{k=1}^K v_k} \sum_{k=1}^K (\text{Precision@k} \times v_k)$$

where $v_k \in \{0, 1\}$ is the relevance binary indicator for chunk $k$, and $\text{Precision@k} = \frac{\sum_{j=1}^k v_j}{k}$.

Notice that if a relevant chunk is placed at **Rank 1**, Precision@1 = 1.0. But if two distractors appear first and the relevant chunk is at **Rank 3**, Precision@3 = 0.333!


In [ ]:
query = "How does HNSW perform fast nearest neighbor search?"
ground_truth = "HNSW builds a hierarchical multi-layer graph with logarithmic search complexity."

rel_chunk = "HNSW organizes vectors into a hierarchy of proximity graphs, achieving logarithmic search complexity O(log N)."
distractor_1 = "The Amazon rainforest covers approximately 5.5 million square kilometers in South America."
distractor_2 = "Sourdough bread relies on wild yeast and lactobacilli fermentation."

# Experiment 1: High Precision (Relevant chunk at Rank 1)
contexts_top = [rel_chunk, distractor_1, distractor_2]
prec_top, flags_top = compute_context_precision(query, contexts_top, ground_truth)

# Experiment 2: Low Precision (Relevant chunk buried at Rank 3)
contexts_buried = [distractor_1, distractor_2, rel_chunk]
prec_buried, flags_buried = compute_context_precision(query, contexts_buried, ground_truth)

print(f"Scenario 1 (Relevant at Rank 1): Context Precision = {prec_top:.4f} (Flags: {flags_top})")
print(f"Scenario 2 (Relevant at Rank 3): Context Precision = {prec_buried:.4f} (Flags: {flags_buried})")


### Visualizing Context Precision Rank-Order Penalization

The chart below shows how Context Precision drops as irrelevant distractor chunks push relevant chunks down the ranking order:


In [ ]:
decay_img_path = "assets/diagrams/track04_context_precision_decay.png"
if os.path.exists(decay_img_path):
    img = plt.imread(decay_img_path)
    plt.figure(figsize=(9, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
else:
    print(f"Plot asset not found at {decay_img_path}")


---

### 4.2 Context Recall (Ground-Truth Coverage)

**Definition:** Measures the fraction of factual propositions in the ground truth that can be directly attributed to the retrieved context chunks.

$$\text{Context Recall} = \frac{|\text{Attributed Ground Truth Claims}|}{|\text{Total Ground Truth Claims}|}$$

Low context recall signals that the retriever failed to find critical evidence, leading to either an incomplete answer or a forced refusal.


In [ ]:
gt_answer = (
    "PyMuPDF parses PDF text and extracts document metadata cleanly, "
    "and ChromaDB indexes the resulting embeddings for nearest neighbor search."
)

# Incomplete retrieval: misses the ChromaDB claim
incomplete_contexts = [
    "PyMuPDF parses PDF text and extracts document metadata cleanly.",
    "Bananas are a rich dietary source of potassium and vitamin B6.",
]

recall_score, verifications = compute_context_recall(gt_answer, incomplete_contexts)
print(f"Context Recall Score: {recall_score:.4f}")
print("Claim Attributions:")
for v in verifications:
    status = "SUPPORTED" if v.is_supported else "MISSING"
    print(f"  [{status}] Claim: '{v.claim}' (Confidence: {v.confidence:.3f})")


---

### 4.3 Faithfulness / Groundedness (Hallucination Detection)

**Definition:** Measures the proportion of atomic claims in the generated response that can be directly inferred from and validated against the retrieved context passages.

$$\text{Faithfulness} = \frac{|\text{Supported Answer Claims}|}{|\text{Total Answer Claims}|}$$

If the generator makes 4 claims and 1 of them cannot be supported by the retrieved context, Faithfulness = 3/4 = 0.75.


In [ ]:
clean_contexts = [
    "HNSW constructs a multi-layer graph with geometric decay factor mL = 1 / ln(M).",
    "Top layers have long-range skip edges for logarithmic search complexity.",
]

# Grounded answer: every claim is supported
grounded_response = (
    "HNSW builds a multi-layer graph, and top layers have long-range skip edges [Doc 2]."
)

# Hallucinated answer: invents GPU cluster training
hallucinated_response = (
    "HNSW builds a multi-layer graph, and it automatically trains a transformer neural network on GPU clusters."
)

faith_clean, verifs_clean = compute_faithfulness(grounded_response, clean_contexts)
faith_hallucinated, verifs_hall = compute_faithfulness(hallucinated_response, clean_contexts)

print(f"Grounded Response Faithfulness:     {faith_clean:.4f}")
print(f"Hallucinated Response Faithfulness: {faith_hallucinated:.4f}")
print("\nHallucinated Claim Breakdown:")
for v in verifs_hall:
    status = "GROUNDED" if v.is_supported else "HALLUCINATION"
    print(f"  [{status}] '{v.claim}' -> Reason: {v.reason}")


---

### 4.4 Answer Relevance (Semantic Query-Response Alignment)

**Definition:** Evaluates whether the generated response directly addresses the user's inquiry, heavily penalizing evasive answers, off-topic hallucinations, and generic boilerplate.


In [ ]:
query_target = "What is the primary role of cross-encoder rerankers in RAG?"

on_topic_response = (
    "Cross-encoder rerankers perform joint cross-attention over query and document tokens to eliminate false positives."
)
off_topic_response = (
    "Mount Everest is the highest mountain peak above sea level in the Himalayas."
)

rel_on = compute_answer_relevance(query_target, on_topic_response)
rel_off = compute_answer_relevance(query_target, off_topic_response)

print(f"On-Topic Response Relevance:  {rel_on:.4f}")
print(f"Off-Topic Response Relevance: {rel_off:.4f}")


## 5. The RAG Triad Composite Score & Pipeline Diagnostics

The **RAG Triad** (TruLens formulation) combines:
1. **Context Relevance (Query <-> Context)**
2. **Groundedness / Faithfulness (Context <-> Answer)**
3. **Answer Relevance (Query <-> Answer)**

### Why Harmonic Mean?
A simple arithmetic average allows high scores on one dimension to mask catastrophic failure on another:
- Example: Context Relevance = 1.0, Answer Relevance = 1.0, Faithfulness = 0.0 (Pure Hallucination).
- Arithmetic Mean: $(1.0 + 1.0 + 0.0) / 3 = 0.67$ (misleadingly passing).
- **Harmonic Mean:** $3 / (1/1.0 + 1/1.0 + 1/0.05) = 0.136$ (**collapses correctly**).


In [ ]:
harmonic_clean = compute_rag_triad_score(1.0, 1.0, 1.0)
harmonic_hallucinated = compute_rag_triad_score(1.0, 0.05, 1.0)

print(f"Healthy Pipeline Harmonic Triad Score:      {harmonic_clean:.4f}")
print(f"Hallucinated Pipeline Harmonic Triad Score: {harmonic_hallucinated:.4f}")


### Radar Fingerprinting of RAG Pipelines

The radar chart below visualizes the distinct metric fingerprints of a healthy RAG system versus an ungrounded system suffering from hallucinations:


In [ ]:
radar_img_path = "assets/diagrams/track04_evaluation_radar.png"
if os.path.exists(radar_img_path):
    img = plt.imread(radar_img_path)
    plt.figure(figsize=(7, 7))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
else:
    print(f"Plot asset not found at {radar_img_path}")


## 6. End-to-End Evaluation Test Suite & Root Cause Analysis

We now run our complete `RAGEvaluationHarness` over four representative test cases representing common production pipeline conditions:
1. **Case A (Healthy):** High precision, high recall, fully grounded synthesis.
2. **Case B (Hallucination):** Complete context retrieval, but generator fabricates unsupported claims.
3. **Case C (Context Pollution):** Relevant evidence retrieved, but buried under irrelevant distractor chunks.
4. **Case D (Incomplete Retrieval):** Retriever misses key facts required by the ground truth.


In [ ]:
harness = RAGEvaluationHarness()

test_dataset = [
    EvaluationSample(
        sample_id="case_a_healthy",
        query="What is the advantage of Cross-Encoder reranking?",
        contexts=[
            "Cross-Encoders resolve bi-encoder representation bottlenecks by applying joint attention over all token pairs.",
            "Bi-encoders generate embeddings independently without cross-token interaction.",
        ],
        ground_truth="Cross-encoders perform full cross-attention over query and document tokens to resolve bi-encoder bottlenecks.",
        generated_answer="Cross-Encoders eliminate bi-encoder representation bottlenecks through joint attention across tokens [Doc 1].",
    ),
    EvaluationSample(
        sample_id="case_b_hallucination",
        query="What is the advantage of Cross-Encoder reranking?",
        contexts=[
            "Cross-Encoders resolve bi-encoder representation bottlenecks by applying joint attention over all token pairs.",
        ],
        ground_truth="Cross-encoders perform full cross-attention over query and document tokens.",
        generated_answer="Cross-Encoders apply joint attention [Doc 1], and they automatically deploy GPU quantum clusters in Tokyo.",
    ),
    EvaluationSample(
        sample_id="case_c_pollution",
        query="How does ChromaDB persist collections?",
        contexts=[
            "The Great Wall of China is an ancient series of fortifications.",
            "Photosynthesis powers terrestrial plant ecosystems.",
            "ChromaDB persists collections locally using SQLite metadata tables and parquet index files.",
        ],
        ground_truth="ChromaDB persists collections locally using SQLite metadata tables and parquet index files.",
        generated_answer="ChromaDB persists collections locally using SQLite and parquet index files [Doc 3].",
    ),
    EvaluationSample(
        sample_id="case_d_incomplete",
        query="Compare PyMuPDF and ChromaDB capabilities.",
        contexts=[
            "PyMuPDF extracts text, images, and tables from PDF documents.",
        ],
        ground_truth="PyMuPDF extracts text from PDFs, while ChromaDB indexes embedding vectors for vector search.",
        generated_answer="PyMuPDF extracts text, images, and tables from PDF documents [Doc 1].",
    ),
]

evaluation_summary = harness.evaluate_dataset(test_dataset)

print(f"Total Evaluated Test Cases: {evaluation_summary['total_samples']}")
print(f"Mean Context Precision:    {evaluation_summary['mean_context_precision']}")
print(f"Mean Context Recall:       {evaluation_summary['mean_context_recall']}")
print(f"Mean Faithfulness:         {evaluation_summary['mean_faithfulness']}")
print(f"Mean Answer Relevance:     {evaluation_summary['mean_answer_relevance']}")
print(f"Mean RAG Triad Score:      {evaluation_summary['mean_rag_triad_score']}")


### Detailed Per-Case Diagnostics Audit

Below, we inspect the automated diagnostic findings for each evaluated test case:


In [ ]:
for res in evaluation_summary["results"]:
    print(f"--- Sample: {res['sample_id']} ---")
    print(f"  Precision: {res['context_precision']} | Recall: {res['context_recall']} | Faithfulness: {res['faithfulness']} | Relevance: {res['answer_relevance']}")
    print(f"  Triad Score: {res['rag_triad_score']}")
    print(f"  Status: {res['diagnostics']['pipeline_status']}")
    print(f"  Diagnostics: {res['diagnostics']['failure_modes']}")
    print()


## 7. Production Architectural Summary & Takeaways

| Dimension | Measured By | Target Metric | Root Cause of Failure | Corrective Action |
|---|---|---|---|---|
| **Retrieval Relevance** | Context Precision | MAP@K $\ge 0.85$ | Unranked candidate pool, noisy embeddings | Add Cross-Encoder Reranker, calibrate top-$k$ |
| **Retrieval Coverage** | Context Recall | Recall $\ge 0.90$ | Semantic gap, chunk boundary cuts | Implement Query Expansion, Sub-Query Decomposition, HyDE |
| **Grounded Synthesis** | Faithfulness | Score $\ge 0.95$ | Weak system prompt, low temperature drift | Strict negative constraints, epistemic refusal, citation enforcement |
| **User Intent Alignment** | Answer Relevance | Cosine $\ge 0.85$ | Evasive boilerplate, off-topic drift | Enforce structured output schemas, task-specific system prompts |

### Key Architectural Takeaways
1. **Never Evaluate Paragraphs Whole:** Always decompose into atomic factual propositions before verifying entailment.
2. **Context Precision is Rank-Sensitive:** Simply retrieving the relevant chunk is not enough; placing it at Rank 3 rather than Rank 1 degrades generator reasoning.
3. **Epistemic Refusal is 100% Faithful:** Refusing to answer when context is absent is a hallmark of a robust, trustworthy system.
4. **Use Harmonic Triad Scoring:** Arithmetic averages mask silent hallucinations. The harmonic mean forces all three pillars to remain sound.
